# Scenario 1: CHP + backup boiler + thermal storage + grid import/export
**Master thesis (EPFL x Voltiris): *A Path to Greenhouse Self-Sufficiency: Heat and CO₂***

This notebook only **runs** the model and shows the results. It follows the same steps as `01_baseline.ipynb`. The model lives in the `ghmodel` package (`src/ghmodel/`):

| Step | Where the code is |
|---|---|
| Parameters and Excel column mapping | `config.py` |
| Read and check the hourly (or daily) Excel | `data.py` |
| Component sizes and cost factors | `sizing.py` |
| Building blocks (CHP, grid export, CHP priority, boiler, storage, grid, sinks) | `blocks.py`, documented in `docs/blocks/` |
| Scenario 1 network | `scenarios/s1_chp.py` |
| Solve, extract results, KPIs, export | `workflow.py`, `results.py` |
| Storage value, financial analysis, scenario comparison | `analysis.py` |
| Plots | `plots.py` |

**Scenario 1** (see `docs/Scenarios.png`):
- A gas-engine **CHP** produces electricity, heat and CO₂ in fixed ratios.
- A **backup boiler** (same block as the Baseline boiler) covers what the CHP cannot.
- The **thermal storage** buffers heat from both units.
- Surplus electricity is **exported** to the grid and counted as revenue. Grid **import** covers the rest.

**CHP priority (merit order):** the CHP is dispatched before the backup boiler and before grid import. This is done with an internal penalty (`priority_penalty_EUR_per_MWh`) on boiler heat and grid import. It steers the dispatch only and is **not** included in any reported cost. Set `chp_priority = False` for pure cost-optimal dispatch.

Units: MW, MWh, t CO₂, EUR. If the Excel file is missing, a **synthetic dummy dataset** (with an export price column) is created so the notebook runs, its numbers mean nothing.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# find the project root (the folder that contains src/ghmodel) so the notebook runs from anywhere
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "ghmodel").exists())
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

In [ ]:
from dataclasses import replace

import pandas as pd
from IPython.display import display

from ghmodel import Params, INPUT_COLUMNS, OPTIONAL_COLUMNS
from ghmodel.data import make_dummy_input, load_inputs, input_overview
from ghmodel.sizing import resolve_sizes
from ghmodel.scenarios import baseline, s1_chp
from ghmodel.workflow import solve, run_scenario
from ghmodel.results import extract_results, check_balances, summarise, export_results
from ghmodel.analysis import storage_value, financial_analysis, compare_scenarios
from ghmodel import plots

pd.options.display.float_format = "{:,.2f}".format

## 2. Configuration: files and plot options
Same columns as the Baseline input, plus **one new column** `elec_export_price_EUR_per_MWh`: the price paid for electricity sold to the grid (EUR/MWh). An example file is in `data/s1_input_data_EXAMPLE_dummy.xlsx`. Column headers and unit factors are defined in `src/ghmodel/config.py` (`INPUT_COLUMNS`, `OPTIONAL_COLUMNS`).

In [ ]:
INPUT_FILE  = ROOT / "data" / "s1_input_data.xlsx"         # put your real data here (with the export price column)
SHEET_NAME  = "hourly"
TIME_COL    = "timestamp"                  # set None if the sheet has no datetime column
START_TIMESTAMP = "2023-01-01 00:00"       # used only if TIME_COL is None / not found
GENERATE_DUMMY_IF_MISSING = True

RESULTS_DIR  = ROOT / "results" / "s1_chp"
SAVE_FIGURES = True
FIG_DPI      = 150
ZOOM_WINTER_START  = None     # e.g. "2023-01-16"; None = coldest week
ZOOM_SUMMER_START  = None     # e.g. "2023-07-10"; None = week with lowest heat demand
ZOOM_STORAGE_START = None     # e.g. "2023-05-08"; None = week with highest storage use

RUN_PRIORITY_CHECK = True     # section 10.3: also solve S1 without the CHP priority (one extra solve)

FIG_DIR = RESULTS_DIR / "figures" if SAVE_FIGURES else None
plots.apply_style(FIG_DPI)

## 3. Model parameters
Values marked *placeholder* in `config.py` are typical orders of magnitude and **must be replaced with sourced data**. `None` means "size automatically from the data" or "use the country default". The Baseline parameters are kept, and the CHP, export and priority parameters are added.

In [ ]:
from ghmodel.config import COUNTRY_DEFAULTS

# ---- Country: sets the default carbon levy on gas and the grid emission factor
COUNTRY = "CH"                            # CH / FR / NL

# Country values (placeholders, defaults from config.py). Edit here to change them for this run only,
# or set co2_price_EUR_per_t / grid_co2_factor_t_per_MWh in Params below to override the country value.
COUNTRY_DEFAULTS["CH"].update(co2_levy_EUR_per_t=125.0, grid_ef_t_per_MWh=0.10)
COUNTRY_DEFAULTS["FR"].update(co2_levy_EUR_per_t=44.6,  grid_ef_t_per_MWh=0.05)
COUNTRY_DEFAULTS["NL"].update(co2_levy_EUR_per_t=0.0,   grid_ef_t_per_MWh=0.30)

display(pd.DataFrame(COUNTRY_DEFAULTS).T.rename(columns={"co2_levy_EUR_per_t": "Carbon levy [EUR/t CO2]",
                                                         "grid_ef_t_per_MWh": "Grid emission factor [t CO2/MWh]"}))

In [ ]:
P = Params(
    # ---- Greenhouse
    greenhouse_area_m2 = 10_000,
    country            = COUNTRY,         # set in the cell above

    # ---- Natural gas (CHP and backup boiler)
    gas_co2_factor_t_per_MWh = 0.202,     # t CO2 per MWh fuel (LHV)
    co2_price_EUR_per_t      = None,      # carbon levy on gas (EUR/t); None -> country value above

    # ---- CHP (gas engine)
    chp_capacity_MW_el       = None,      # None -> chp_heat_share_of_peak x peak heat demand, as MW electric
    chp_heat_share_of_peak   = 0.5,       # placeholder
    chp_eta_el               = 0.42,      # LHV, placeholder
    chp_eta_th               = 0.45,      # LHV, placeholder
    chp_co2_usable_fraction  = 1.0,       # share of exhaust CO2 usable after cleaning, placeholder
    chp_capex_EUR_per_kW_el  = 900,       # placeholder
    chp_lifetime_y           = 15,        # placeholder
    chp_fom_frac             = 0.02,      # placeholder
    chp_vom_EUR_per_MWh_el   = 12.0,      # placeholder
    chp_priority             = True,      # CHP before backup boiler and grid import
    priority_penalty_EUR_per_MWh = 200.0, # dispatch weight only, not a cost; must be < voll_heat_EUR_per_MWh

    # ---- Backup boiler (same block as the Baseline boiler)
    boiler_eta                 = 0.92,    # LHV, placeholder
    boiler_capacity_MW_th      = None,    # None -> boiler_sizing_margin x peak heat demand (full backup)
    boiler_sizing_margin       = 1.2,
    boiler_co2_usable_fraction = 1.0,     # placeholder
    boiler_capex_EUR_per_kW_th = 80,      # placeholder
    boiler_lifetime_y          = 20,
    boiler_fom_frac            = 0.02,
    boiler_ramp_limit_per_h    = 0.5,     # None = off, placeholder

    # ---- Thermal storage
    tes_capacity_MWh         = None,      # None -> tes_hours_of_peak x peak heat demand
    tes_hours_of_peak        = 6,
    tes_power_MW_th          = None,      # None -> boiler thermal capacity
    tes_hours_at_full_power  = 4.0,       # None = off, placeholder
    tes_min_soc_frac         = 0.1,       # 0 = off, placeholder
    tes_eta_charge           = 0.98,      # placeholder
    tes_eta_discharge        = 0.98,      # placeholder
    tes_standing_loss_per_h  = 0.001,     # placeholder
    tes_capex_EUR_per_kWh    = 10,        # placeholder
    tes_lifetime_y           = 25,
    tes_fom_frac             = 0.01,

    # ---- Grid (import and export)
    grid_import_limit_MW           = None,
    grid_export_limit_MW           = None,   # None -> no export limit
    grid_tariff_adder_EUR_per_MWh  = 0.0,
    grid_co2_factor_t_per_MWh      = None,   # None -> country value above

    # ---- Economics
    discount_rate      = 0.05,
    project_lifetime_y = 20,              # horizon of the financial analysis (section 10)

    # ---- Size optimisation: False = fixed input, True = decision variable
    OPTIMISE_BOILER      = False,
    OPTIMISE_TES         = False,
    OPTIMISE_CHP         = False,
    tes_capacity_max_MWh = None,

    # ---- Assumed sinks (not drawn in the figure): 0 = free
    heat_dump_cost_EUR_per_MWh = 0.0,
    co2_vent_cost_EUR_per_t    = 0.0,

    # ---- Slack (numerical safety, should stay at 0)
    voll_heat_EUR_per_MWh = 1_000,
    voll_co2_EUR_per_t    = 5_000,

    solver_name = "highs",                # solver_options: see config.py (HiGHS defaults, 1 h time limit)
)
print(f"Country {P.country}: carbon levy {P.co2_levy():.1f} EUR/t CO2, grid emission factor {P.grid_ef():.3f} t CO2/MWh")

## 4. Input data

In [ ]:
if not INPUT_FILE.exists():
    if GENERATE_DUMMY_IF_MISSING:
        make_dummy_input(INPUT_FILE, SHEET_NAME, export_price=True)
        print(f"'{INPUT_FILE.name}' not found -> synthetic dummy file created. Replace it with real data.")
    else:
        raise FileNotFoundError(INPUT_FILE)

data = load_inputs(INPUT_FILE, sheet_name=SHEET_NAME, time_col=TIME_COL, start_timestamp=START_TIMESTAMP,
                   params=P, input_columns=INPUT_COLUMNS, optional_columns=OPTIONAL_COLUMNS)
if "elec_export_price_EUR_MWh" not in data:
    raise KeyError("Scenario 1 needs the column 'elec_export_price_EUR_per_MWh' in the input file.")
print(f"{len(data)} hourly values, {data.index[0]} -> {data.index[-1]}")
display(input_overview(data, P.greenhouse_area_m2))
print(f"Export price: mean {data.elec_export_price_EUR_MWh.mean():.1f} EUR/MWh "
      f"(import price mean {data.elec_price_EUR_MWh.mean():.1f} EUR/MWh)")

In [ ]:
plots.plot_inputs(data, out_dir=FIG_DIR)

## 5. Build the network

In [ ]:
sizes = resolve_sizes(data, P)
display(sizes.table(include_chp=True))
print(f"Sizes optimised: boiler {P.OPTIMISE_BOILER} | storage {P.OPTIMISE_TES} | CHP {P.OPTIMISE_CHP} | "
      f"CHP priority: {P.chp_priority}")

n = s1_chp.build(data, P, sizes)
print(n)

## 6. Solve: minimise operating cost
With `chp_priority = True` the solver objective also contains the priority penalty, which is a dispatch weight. The real costs are computed in section 7 without it.

In [ ]:
solve(n, P)

## 7. Results and KPIs
Hourly results `r` have all flows positive (MW, t/h, MWh, EUR). The balance check must pass and the unserved demand must be zero. The KPI table has the same rows as in the Baseline, so the two scenarios can be compared row by row.

In [ ]:
r = extract_results(n, data, P)
check_balances(r)
res = summarise(n, r, data, P, sizes)
print(f"Priority penalty in the objective (not a cost): {r.priority_penalty_EUR.sum():,.0f} EUR")

display(res.summary)
display(res.heat_balance.to_frame())

In [ ]:
display(res.monthly)

## 8. Plots

### 8.1 Heat: yearly view

In [ ]:
plots.plot_heat_yearly(res, out_dir=FIG_DIR)

### 8.2 Three hourly weeks (heat, storage, electricity, CO₂, prices)

In [ ]:
plots.plot_zoom_weeks(res, zoom_winter_start=ZOOM_WINTER_START, zoom_summer_start=ZOOM_SUMMER_START,
                      zoom_storage_start=ZOOM_STORAGE_START, out_dir=FIG_DIR)

### 8.3 Thermal storage

In [ ]:
plots.plot_storage(res, out_dir=FIG_DIR)

### 8.4 CO₂

In [ ]:
plots.plot_co2(res, out_dir=FIG_DIR)

### 8.5 Fuel, grid import/export and monthly cost

In [ ]:
plots.plot_fuel_grid_cost(res, out_dir=FIG_DIR)

### 8.6 Duration curves

In [ ]:
plots.plot_duration(res, out_dir=FIG_DIR)

### 8.7 Heatmaps

In [ ]:
plots.plot_heatmaps(res, out_dir=FIG_DIR)

### 8.8 Cost breakdown and fate of the produced heat

In [ ]:
plots.plot_cost_and_heat_fate(res, out_dir=FIG_DIR)

## 9. Value of the thermal storage
The same Scenario 1 is solved a second time **without storage** (capacity and power 0). Everything else is unchanged, including the CHP priority. **Saved by storage = cost without storage − cost with storage.** In S1, the storage lets the CHP run in more hours and shifts CHP heat to when it is needed.

In [ ]:
tes_table, tes_indicators, res_no_tes = storage_value(s1_chp.build, data, P, res)
display(tes_table)
display(tes_indicators)
plots.plot_storage_savings(tes_table, tes_indicators, out_dir=FIG_DIR)

## 10. Comparison with the Baseline and financial analysis
### 10.1 KPI comparison
The Baseline is solved with the **same input data and parameters**, so the difference comes only from the CHP and the grid export.

In [ ]:
n_base, res_base = run_scenario(baseline.build, data, P, verbose=False)
comparison = compare_scenarios({"Baseline": res_base, "S1": res})
comparison["S1 - Baseline"] = comparison["S1"] - comparison["Baseline"]
display(comparison)

### 10.2 Payback, NPV and IRR of the S1 equipment against the Baseline
The equipment of S1 (the greenhouse is excluded) is the CHP, the backup boiler and the thermal storage. The Baseline already has the boiler and the storage at the same sizes, so the **additional investment is the CHP**. The table `investments` lists the full equipment set of S1.

Cash flows over `project_lifetime_y` years:
- **Year 0:** the additional investment.
- **Every year:** the operating cost saving (gas, levy, electricity purchase, **export revenue**, CHP variable O&M) minus the additional fixed O&M.
- **Replacements** at the end of each lifetime (CHP 15 y), and a straight-line salvage value at the horizon.

Sensitivity: ±20 % on prices and investment, first order (the dispatch is **not** re-optimised).

In [ ]:
fin = financial_analysis(res, res_base, P, name="S1", ref_name="Baseline")
print("Equipment of S1 (overnight investment, lifetime, fixed O&M):")
display(fin["investments"])
display(fin["indicators"])
display(fin["sensitivity"])
plots.plot_financials(fin, out_dir=FIG_DIR)

### 10.3 Cost of the CHP priority rule
The priority forces the CHP to run before the boiler and grid import even in hours where that is not the cheapest choice. Solving S1 again without the priority shows what the rule costs, and how many hours the CHP would run on pure economics.

In [ ]:
if RUN_PRIORITY_CHECK:
    _, res_free = run_scenario(s1_chp.build, data, replace(P, chp_priority=False), verbose=False)
    rows = ["CHP full-load hours", "CHP electricity output", "Grid export", "Grid import", "Boiler heat output",
            "Operating cost (OPEX, net of export revenue)", "Total annualised cost", "Total CO2"]
    prio_table = compare_scenarios({"S1 with priority": res, "S1 cost-optimal (no priority)": res_free})
    prio_table = prio_table.droplevel(0).loc[rows]
    prio_table["Cost of the rule"] = prio_table.iloc[:, 0] - prio_table.iloc[:, 1]
    display(prio_table)

## 11. Export

In [ ]:
export_results(res, n, RESULTS_DIR, name="s1_chp")